# ポートフォリオゲーム ボラティリティ分析

**yfinance**（Yahoo Finance）で指定30銘柄の株価・ボラティリティ・相関を分析。

## 前提
認証不要・APIキー不要・無料。
```bash
pip install yfinance pandas numpy matplotlib seaborn jupyter
```
東証銘柄は **コード + `.T`** で取得（例: 任天堂 → `7974.T`）。

## 1. セットアップ

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# matplotlibで日本語表示
plt.rcParams['font.family'] = ['Hiragino Sans', 'Yu Gothic', 'Meiryo', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
print('ready')

## 2. 指定30銘柄を読み込み

In [ ]:
stocks = pd.read_csv(ROOT / 'data' / 'stocks.csv')
stocks['ticker'] = stocks['code'].astype(str).str.zfill(4) + '.T'  # 東証 = .T
print(f'{len(stocks)} 銘柄ロード')
stocks.head()

## 3. 日次株価データ取得（過去1年）

In [ ]:
tickers = stocks['ticker'].tolist()
to_date = datetime.today()
from_date = to_date - timedelta(days=365)
print(f'期間: {from_date.date()} 〜 {to_date.date()}')

raw = yf.download(
    tickers,
    start=from_date.strftime('%Y-%m-%d'),
    end=to_date.strftime('%Y-%m-%d'),
    auto_adjust=True,   # 分割・配当調整済み価格
    progress=False,
    group_by='ticker',
)

# ティッカー → 社名へリネーム
ticker_to_name = dict(zip(stocks['ticker'], stocks['name']))
close = pd.concat({ticker_to_name[t]: raw[t]['Close'] for t in tickers if t in raw.columns.get_level_values(0)}, axis=1)
close = close.dropna(how='all').sort_index()
print(f'価格パネル shape={close.shape}')
close.tail()

## 4. 日次リターン

In [ ]:
returns = close.pct_change().dropna(how='all')
print(f'リターン shape={returns.shape}')
returns.tail()

## 5. 年率ボラティリティ / 年率リターン / シャープ
$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$

In [ ]:
TRADING_DAYS = 252
annual_vol = returns.std() * np.sqrt(TRADING_DAYS)
annual_ret = returns.mean() * TRADING_DAYS
sharpe = annual_ret / annual_vol  # 無リスク金利 0% 想定

name_to_sector = dict(zip(stocks['name'], stocks['sector']))
summary = pd.DataFrame({
    'sector': annual_vol.index.map(name_to_sector),
    'annual_return': annual_ret,
    'annual_volatility': annual_vol,
    'sharpe': sharpe,
}).sort_values('annual_volatility', ascending=False)
summary.style.format({'annual_return': '{:.2%}', 'annual_volatility': '{:.2%}', 'sharpe': '{:.2f}'})

## 6. 可視化

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
colors = {'エンタメ': '#22c55e', '運輸': '#3b82f6', '小売': '#10b981'}
ordered = summary.sort_values('annual_volatility')
ax.barh(ordered.index, ordered['annual_volatility'], color=[colors[s] for s in ordered['sector']])
ax.set_xlabel('年率ボラティリティ')
ax.set_title('指定30銘柄 — 年率ボラティリティ（過去1年）')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=s) for s, c in colors.items()])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
for sector, group in summary.groupby('sector'):
    ax.scatter(group['annual_volatility'], group['annual_return'], label=sector, s=80, alpha=0.7, color=colors[sector])
    for name, row in group.iterrows():
        ax.annotate(name, (row['annual_volatility'], row['annual_return']), fontsize=8)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('年率ボラティリティ')
ax.set_ylabel('年率リターン')
ax.set_title('リスク・リターン散布図')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
corr = returns.corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, annot=False, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('指定30銘柄 — 日次リターン相関行列')
plt.tight_layout()
plt.show()

## 7. ローリング・ボラ（20日 ≒ 1ヶ月）

In [ ]:
WINDOW = 20
rolling_vol = returns.rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, sector in zip(axes, ['エンタメ', '運輸', '小売']):
    cols = [n for n in rolling_vol.columns if name_to_sector.get(n) == sector]
    rolling_vol[cols].plot(ax=ax, legend=False, alpha=0.7)
    ax.set_title(f'{sector} — 20日ローリング年率ボラ')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend(fontsize=7, ncol=5, loc='upper left')
plt.tight_layout()
plt.show()

## 8. データ書き出し（任意）

In [ ]:
OUT = ROOT / 'analysis' / 'output'
OUT.mkdir(exist_ok=True)
summary.to_csv(OUT / 'summary_metrics.csv')
close.to_csv(OUT / 'close_prices.csv')
returns.to_csv(OUT / 'daily_returns.csv')
print('saved to', OUT)